# nuReasoning data & reasoning visualization

This tutorial walks through obtaining the dataset and visualizing both the sensor data and the reasoning annotations.

Prerequisites: install the devkit (`conda env create -f environment.yml && pip install -e .`). The first code cell `chdir`s to the repository root. Log in to Hugging Face (`hf auth login`) only when downloading; set `NUREASONING_VALIDATION_ROOT` to reuse an already extracted validation directory.

In [ ]:
import os
from pathlib import Path

_here = Path.cwd()
REPO_ROOT = _here if (_here / "nureasoning").is_dir() else _here.parent
os.chdir(REPO_ROOT)

DATASET_DIR = Path(os.environ.get("NUREASONING_DATASET_DIR", "./dataset"))
VALIDATION_ROOT = Path(os.environ.get(
    "NUREASONING_VALIDATION_ROOT",
    str(DATASET_DIR / "data" / "validation" / "part_1"),
))
VIZ_OUTPUT_ROOT = Path(os.environ.get("NUREASONING_VIZ_OUTPUT", "."))
MAX_CLIPS = int(os.environ.get("NUREASONING_MAX_CLIPS", "1"))

print("REPO_ROOT", REPO_ROOT)
print("VALIDATION_ROOT", VALIDATION_ROOT, "exists", VALIDATION_ROOT.is_dir())

if VALIDATION_ROOT.is_dir():
    print(f"Using existing extracted data: {VALIDATION_ROOT}")
else:
    print("Validation data not found locally; downloading and extracting it.")
    !python -m nureasoning.dataset.download --local-dir "{DATASET_DIR}" --splits validation --max-workers 16

## Sensor data

`nureasoning.visualization.view_data` writes a composite overview: an 8-camera grid with an ego-state card, a BEV panel (HD map, actors, ego history/future, route, traffic lights), and an optional LiDAR top-down view. Frame 100 is the key frame (t = 10 s) used by the planning benchmark and the challenge. Use `--video` for an MP4, or `--save-panels` for per-reasoning-frame PNG dumps.

In [ ]:
!python -m nureasoning.visualization.view_data \
    --input-root "{VALIDATION_ROOT}" \
    --frame-index 100 \
    --max-clips {MAX_CLIPS} \
    --output-dir "{VIZ_OUTPUT_ROOT / 'nureasoning_viz'}"

## Reasoning annotations

`nureasoning.visualization.view_reasoning` pretty-prints the structured **Spatial**, **Decision**, and **Counterfactual** reasoning of each frame. Spatial reasoning is annotated on every reasoning frame (~1 Hz); Decision and Counterfactual text is attached every 5 s, including the key frame (t ≈ 10 s, frame 100). `--frame-index` snaps to the nearest annotated frame. `--save-figures` writes the Driving / Counterfactual text above the left, front, and right cameras with spatial 2D boxes; `--no-spatial` keeps the printed output short. `--video` writes an MP4 with history frames, a paused reasoning frame, and future frames. Without `--frame-index`, `--stride` samples every Nth **reasoning-annotated** frame.

In [ ]:
REASONING_VIZ_DIR = VIZ_OUTPUT_ROOT / "reasoning_viz"
!python -m nureasoning.visualization.view_reasoning --input-root "{VALIDATION_ROOT}" \
    --max-clips {MAX_CLIPS} --frame-index 100 --no-spatial --save-figures --output-dir "{REASONING_VIZ_DIR}"

In [ ]:
# Display one of the saved reasoning figures inline.
from IPython.display import Image, display

figures = sorted(REASONING_VIZ_DIR.glob("*/*.jpg"))
if figures:
    display(Image(filename=str(figures[0])))
else:
    print("No figures found; run the previous cell first.")